# Variable selection using RF


In [ ]:
import pathlib

import geopandas as gpd
import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import shapely
from esda.getisord import G_Local
from esda.join_counts import Join_Counts
from esda.join_counts_local import Join_Counts_Local
from esda.moran import Moran_Local
from libpysal import graph
from sklearn import ensemble, metrics, model_selection
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler

## Preprocess the data
Remove unnecessary column, devide the data by total population and standardize it.

In [ ]:
def process_file(path, path_total):
    cols_no_division = [
        "Hustota obyvatel na obytnou plochu",
        "Počet obyvatel na dům",
        "Počet obyvatel na byt",
    ]

    # Load total population data
    total = pd.read_csv(path_total, dtype={"nadzsjd": str}, index_col=0)

    # Load main data
    data = gpd.read_parquet(path)

    # Merge data
    data_total = data.join(total)

    # Drop unnecessary columns
    data_relative = data_total.drop(
        columns=[
            "NUTS_2",
            "naz_oblast",
            "NUTS_3",
            "naz_kraj",
            "kod_okres",
            "naz_okres",
            "naz_orp",
            "kod_obec",
            "naz_obec",
            "kod_mco",
            "nazev_mco",
        ]
    )

    # Convert columns (except 'geometry') to float
    cols_numeric = data_relative.columns.drop(["geometry", "kod_orp"])
    data_relative[cols_numeric] = data_relative[cols_numeric].astype(float)

    # Normalize by total population (except certain columns)
    population = data_relative["Obyvatelstvo celkem"].replace(
        0, np.nan
    )  # avoid division by zero
    cols_to_normalize = [
        col
        for col in cols_numeric
        if col not in cols_no_division + ["Obyvatelstvo celkem"]
    ]
    data_relative[cols_to_normalize] = data_relative[cols_to_normalize].div(
        population, axis=0
    )

    # Drop rows with NaNs after division
    data_relative = data_relative.dropna(subset=cols_to_normalize)

    # Clip extremely large or small values
    data_relative[cols_to_normalize] = data_relative[cols_to_normalize].clip(-1e6, 1e6)
    data_relative[cols_no_division] = data_relative[cols_no_division].clip(-1e6, 1e6)

    # Replace any remaining infinities with NaN and drop them
    data_relative.replace([np.inf, -np.inf], np.nan, inplace=True)
    data_relative.dropna(subset=cols_to_normalize + cols_no_division, inplace=True)

    # Scale columns
    #    scaler = StandardScaler()
    #   data_relative[cols_to_normalize] = scaler.fit_transform(data_relative[cols_to_normalize])
    #   data_relative[cols_no_division] = scaler.fit_transform(data_relative[cols_no_division])

    return data_relative

In [ ]:
path = "/data/uscuni-restricted/04_spatial_census/_merged_census_2021.parquet"
path_total = "/data/uscuni-restricted/04_spatial_census/total.csv"

In [ ]:
data_relative = process_file(path, path_total)

In [ ]:
data_r = data_relative[data_relative.columns.drop(["Obyvatelstvo celkem", "kod_orp"])]
data_r.to_parquet(
    "/data/uscuni-restricted/04_spatial_census/_merged_census_2021_relative_scaled.parquet"
)

## Assign cluster label


In [ ]:
clusters = pd.read_csv(
    "/data/uscuni-restricted/04_spatial_census/cluster_assignment_v10.csv",
    dtype={"kod_nadzsj_d": str},
)
cluster_mapping = pd.read_parquet(
    "/data/uscuni-ulce/processed_data/clusters/cluster_mapping_v10.pq"
)
data = data_relative.merge(clusters, left_on="nadzsjd", right_on="kod_nadzsj_d")
variables = data.columns.drop(
    [
        "geometry",
        "kod_nadzsj_d",
        "final_without_noise",
        "kod_orp",
        "Obyvatelstvo celkem",
    ]
)

data["Cluster"] = data["final_without_noise"].map(cluster_mapping[3])

In [ ]:
data["Cluster"].unique()

## Classification


In [ ]:
# Assign independent variables and the target
independent = data[variables]
target = data["Cluster"]

In [ ]:
# Plot original data
ax = data.plot(target, legend=True, figsize=(9, 9), markersize=0.1, categorical=True)
ax.set_axis_off()

In [ ]:
# Split data
X_train, X_test, y_train, y_test = model_selection.train_test_split(
    independent,
    target,
    test_size=0.2,
    random_state=42,
)

In [ ]:
rf_model = RandomForestClassifier(random_state=42, n_jobs=-1, class_weight="balanced")
rf_model.fit(X_train, y_train)
rf_model.score(X_train, y_train), rf_model.score(X_test, y_test)

#### Parameter tuning


In [ ]:
np.mean([estimator.tree_.max_depth for estimator in rf_model.estimators_])

In [ ]:
rf_model = ensemble.RandomForestClassifier(
    min_samples_split=10,
    min_samples_leaf=5,
    max_depth=15,
    n_jobs=-1,
    random_state=42,
    n_estimators=300,
    max_features="log2",
    class_weight="balanced",
)
rf_model.fit(X_train, y_train)

In [ ]:
rf_model.score(X_train, y_train), rf_model.score(X_test, y_test)

In [ ]:
pred = rf_model.predict(X_test)
pred

In [ ]:
proba = rf_model.predict_proba(X_test)
pd.DataFrame(proba, columns=rf_model.classes_, index=X_test.index)

In [ ]:
accuracy = metrics.accuracy_score(pred, y_test)
kappa = metrics.cohen_kappa_score(pred, y_test)

summary = f"""\
Evaluation metrics
==================
Basic model:
  Accuracy: {round(accuracy, 3)}
  Kappa:    {round(kappa, 3)}
"""

print(summary)

In [ ]:
predicted = model_selection.cross_val_predict(
    rf_model, independent, target, cv=4, n_jobs=-1
)

In [ ]:
predicted = model_selection.cross_val_predict(
    rf_model, independent, target, cv=4, n_jobs=-1
)

ax = data.plot(predicted, legend=True, figsize=(9, 9), markersize=0.1, categorical=True)
ax.set_axis_off()

### Residual exploration

In [ ]:
ax = data.plot(
    predicted == target,
    categorical=True,
    figsize=(9, 9),
    markersize=0.1,
    cmap="bwr_r",
    legend=True,
)
ax.set_axis_off()

### Feature importance

In [ ]:
feat_importances = pd.Series(
    rf_model.feature_importances_, index=X_train.columns
).sort_values()
plt.figure(figsize=(5, 20))
plt.axvline(x=0.01, color="red", linestyle="--", linewidth=1)


feat_importances.plot(kind="barh")

In [ ]:
num_high = (feat_importances > 0.01).sum()
print(f"Number of features with importance > 0.05: {num_high}")